# Faruq-v3 — Shape / Aspect-Ratio Conflict Audit

Validation-only descriptive diagnostic. **No training, no inference, no test.** This is the closing audit for the geometry hypothesis track. It asks whether `mixed_geometry` and size-pair classification errors are associated with extreme object shape.

Frozen before reading results: `aspect_ratio = long_side / short_side`; `extreme_shape` means absolute aspect-ratio distance from the pair median greater than one pair IQR. If mixed geometry is not more shape-extreme and errors are not elevated on extreme shapes, close the geometry track rather than building a geometry-conditioned model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-circle-cpe-hard-confusion-reduction-v1/events/CPE0_seed42_events.json',
    'experiments/faruq-v3-circle-cpe-hard-confusion-reduction-v1/events/CIR0_seed42_events.json',
    'experiments/faruq-v3-scale-identifiability-audit-v1/scale_identifiability_audit.json',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
CPE0_EVENT = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
CIR0_EVENT = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
SCALE_JSON = require_project_artifact(PROJECT_ROOT, REQUIRED[3])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT/'faruq_grouped_summary.json').is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT/'experiments/faruq-v3-shape-aspect-ratio-conflict-audit-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY = OUTPUT_ROOT/'shape_aspect_ratio_conflict_audit.json'
print('CPE0 EVENT:', CPE0_EVENT)
print('CIR0 EVENT:', CIR0_EVENT)
print('SCALE AUDIT:', SCALE_JSON)
print('OUTPUT:', SUMMARY)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_shape_aspect_ratio_conflict_audit.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
command = [
    sys.executable, '-m', 'coffee_detector.analysis.shape_aspect_ratio_conflict_audit',
    '--cpe0-event', str(CPE0_EVENT),
    '--cir0-event', str(CIR0_EVENT),
    '--scale-json', str(SCALE_JSON),
    '--data-root', str(DATA_ROOT),
    '--output', str(SUMMARY),
]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
assert result['screening_decision_remains'] == 'STOP_CIRCLE_CPE'


In [ ]:
import pandas as pd
from IPython.display import display

print('GLOBAL GEOMETRY STATES:', result['global_geometry_state_counts_across_pair_memberships'])
print('GLOBAL SHAPE BY STATE:')
print(json.dumps(result['global_shape_by_geometry_state'], indent=2, ensure_ascii=False))
print('GLOBAL MODEL SHAPE-ERROR SUMMARY:')
print(json.dumps(result['global_model_shape_error_summary'], indent=2, ensure_ascii=False))
print('SCREENING DECISION REMAINS:', result['screening_decision_remains'])

rows = []
for pair in result['pairs']:
    assoc = pair['mixed_vs_both_support_gt']
    row = {
        'family': pair['family'],
        'n': pair['gt_instances_in_pair'],
        'AR_AUC_mixed_greater': assoc['aspect_ratio_auc_mixed_greater'],
        'mixed_extreme_rate': assoc['mixed_extreme_shape_rate'],
        'clearGT_extreme_rate': assoc['both_support_gt_extreme_shape_rate'],
        'mixed_minus_clear_extreme_rate': assoc['extreme_rate_difference_mixed_minus_clear'],
    }
    for model in ('CPE0','CIR0'):
        s = pair['models'][model]
        row[f'{model}_errors'] = s['pair_errors_total']
        row[f'{model}_extreme_error_rate'] = s['extreme_shape']['error_rate']
        row[f'{model}_nonextreme_error_rate'] = s['non_extreme_shape']['error_rate']
        row[f'{model}_error_share_extreme'] = s['extreme_shape']['share_of_pair_errors']
    rows.append(row)
display(pd.DataFrame(rows))
print('SUMMARY:', SUMMARY)
